<a href="https://colab.research.google.com/github/Amirtha-Raja-Rajeswari/Amirtha-231501012-_Gen-AI/blob/main/GEN_AI__LAB_EXP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ex1a: Text Generation using N-gram Language model**

In [9]:
import nltk
from nltk.corpus import brown
from nltk import FreqDist, bigrams, trigrams, ConditionalFreqDist
from collections import defaultdict

nltk.download('brown')
nltk.download('universal_tagset')

sentences = brown.sents()

tokens = [word.lower() for sent in sentences for word in sent]
print(f"Total number of tokens: {len(tokens)}")

bigram_cfd = ConditionalFreqDist(bigrams(tokens))
trigram_cfd = ConditionalFreqDist(((w1, w2), w3) for w1, w2, w3 in trigrams(tokens))

def predict_next_word(context, n=5):
    context = [w.lower() for w in context.split()]

    if len(context) == 1:
        word = context[0]
        suggestions = bigram_cfd[word].most_common(n)
    elif len(context) >= 2:
        w1, w2 = context[-2], context[-1]
        suggestions = trigram_cfd[(w1, w2)].most_common(n)
    else:
        return []

    return [word for word, freq in suggestions]

test_words = ["she","he","in a","and","on the"]
for phrase in test_words:
    print(f"Next words for '{phrase}': {predict_next_word(phrase)}")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


Total number of tokens: 1161192
Next words for 'she': ['was', 'had', 'said', 'would', 'could']
Next words for 'he': ['was', 'had', 'said', 'is', 'would']
Next words for 'in a': ['few', 'way', 'long', 'very', 'single']
Next words for 'and': ['the', 'a', 'he', ',', 'in']
Next words for 'on the': ['other', 'basis', 'floor', 'ground', 'part']


**Ex1b: Hidden Markov Model (HMM) based Predictive Text System**

In [13]:
import nltk
from nltk.corpus import brown
from nltk.tag import hmm

nltk.download('brown')
nltk.download('universal_tagset')

sentences = brown.sents()
tokens = [word.lower() for sent in sentences for word in sent]
print(f"Total tokens: {len(tokens)}")

tagged_sents = brown.tagged_sents(tagset='universal')
tagged_sents_lower = [[(word.lower(), tag) for (word, tag) in sent] for sent in tagged_sents]

trainer = hmm.HiddenMarkovModelTrainer()
hmm_model = trainer.train_supervised(tagged_sents_lower)

print("\nSample POS Transition Probabilities (tag -> next tag):")
for tag in ['NOUN', 'VERB', 'DET', 'ADJ']:
    if tag in hmm_model._transitions:
        transition_dist = hmm_model._transitions[tag]
        sorted_transitions = sorted([(sample, transition_dist.prob(sample)) for sample in transition_dist.samples()],
                                    key=lambda item: item[1], reverse=True)
        print(f"{tag}: {[(sample, prob) for sample, prob in sorted_transitions[:3]]}")

print("\nSample Emission Probabilities (tag -> words):")
for tag in ['NOUN', 'VERB', 'DET', 'ADJ']:
    if tag in hmm_model._outputs:
        output_dist = hmm_model._outputs[tag]
        sorted_outputs = sorted([(sample, output_dist.prob(sample)) for sample in output_dist.samples()],
                                key=lambda item: item[1], reverse=True)
        print(f"{tag}: {[(sample, prob) for sample, prob in sorted_outputs[:3]]}")

def predict_next_word(pos_tag, top_n=5):
    """
    Predict next words given a POS tag using HMM emission probabilities
    """
    if pos_tag in hmm_model._outputs:
        output_dist = hmm_model._outputs[pos_tag]
        sorted_outputs = sorted([(word, output_dist.prob(word)) for word in output_dist.samples()],
                                key=lambda item: item[1], reverse=True)
        return [word for word, prob in sorted_outputs[:top_n]]
    else:
        return []

test_tags = ['DET', 'NOUN', 'VERB', 'ADJ']
for tag in test_tags:
    print(f"Predicted words for POS '{tag}': {predict_next_word(tag)}")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


Total tokens: 1161192

Sample POS Transition Probabilities (tag -> next tag):
NOUN: [('.', 0.2846630547181078), ('ADP', 0.24513552089250085), ('VERB', 0.15934446046518402)]
VERB: [('VERB', 0.18432723051990715), ('ADP', 0.1693147474924445), ('DET', 0.16306775874906926)]
DET: [('NOUN', 0.6265501711666338), ('ADJ', 0.23975007481697214), ('VERB', 0.06467836001197072)]
ADJ: [('NOUN', 0.6529453937149002), ('.', 0.10041821006093918), ('ADP', 0.08850519775361453)]

Sample Emission Probabilities (tag -> words):
NOUN: [('time', 0.005795513104319236), ('man', 0.004365687078582367), ('af', 0.003610855064995391)]
VERB: [('is', 0.055310533515731876), ('was', 0.05370725034199726), ('be', 0.03487824897400821)]
DET: [('the', 0.5106445091556645), ('a', 0.1683708098876798), ('his', 0.050773980250914105)]
ADJ: [('other', 0.02032942750325486), ('new', 0.019529150392374673), ('first', 0.012314711959962256)]
Predicted words for POS 'DET': ['the', 'a', 'his', 'this', 'an']
Predicted words for POS 'NOUN': ['ti

**Ex2: Bias Audit of Pre-Trained Word Embeddings Using the Word Embedding**

In [19]:
!pip install gensim
import gensim
import numpy as np
from itertools import combinations
import gensim.downloader as api

print("Loading Word2Vec model. This may take a few minutes...")
model = api.load('word2vec-google-news-300')
print("Model loaded!")

X = ["man", "male", "boy", "gentleman", "he", "him"]
Y = ["woman", "female", "girl", "lady", "she", "her"]
A = ["career", "business", "professional", "office", "salary"]
B = ["family", "home", "children", "parents", "house"]

def cosine_similarity(w1, w2):
    return np.dot(w1, w2)/(np.linalg.norm(w1)*np.linalg.norm(w2))

def association(word, A, B):
    return np.mean([cosine_similarity(model[word], model[a]) for a in A]) - \
           np.mean([cosine_similarity(model[word], model[b]) for b in B])

def weat_effect_size(X, Y, A, B):
    assoc_X = [association(x, A, B) for x in X]
    assoc_Y = [association(y, A, B) for y in Y]
    mean_diff = np.mean(assoc_X) - np.mean(assoc_Y)
    std_dev = np.std(assoc_X + assoc_Y, ddof=1)
    return mean_diff / std_dev

def weat_score(X, Y, A, B):
    return sum(association(x, A, B) for x in X) - sum(association(y, A, B) for y in Y)

score = weat_score(X, Y, A, B)
effect_size = weat_effect_size(X, Y, A, B)

print(f"\nWEAT Score: {score:.4f}")
print(f"Effect Size: {effect_size:.4f}")

if effect_size > 0:
    print("Bias toward X-A association (male-career).")
elif effect_size < 0:
    print("Bias toward Y-A association (female-career).")
else:
    print("No detectable bias.")

Loading Word2Vec model. This may take a few minutes...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Model loaded!

WEAT Score: 0.3028
Effect Size: 0.4755
Bias toward X-A association (male-career).
